# Constraint Plotting: Manual Parameter Exploration

This notebook shows how to generate constraint plots by **manually defining source parameters**, without running waveform propagation.

## Use Cases:
- Theoretical parameter space exploration
- Quick constraint plot generation
- Testing different source configurations

## What you'll get:
- Constraint plot in parameter space

## What you won't get:
- Spectrogram (requires propagation - see `examples_complete_workflow.ipynb`)

In [ ]:
import numpy as np
from inputs.source import Source
from inputs.spectrum import SignalModel
from inputs.experiment import Experiment
from utils.constants import YEAR_TO_SEC, DAY_TO_SEC
from plotting.output_handler import OutputHandler
from plotting.plots import Plot

## Approach 1: Manually Define Source Parameters

Define your source parameters directly without any propagation.

### Example 1: Single Source (1x1 Plot)

In [ ]:
# --- Example 1 Parameters ---

# Source
Etot = 0.01                  # Total energy [solar masses]
mass = 1e-20                 # Scalar field mass [eV]
tstar = 1                    # Burst duration [seconds]
R = 10000.0                  # Distance [parsecs]
ULB_type = 'ALP'             # Type: 'scalar' or 'ALP'
coupling_type = 'electron'   # Coupling: 'photon', 'electron', 'gluon'
coupling_order = 'quad'      # Order: 'linear' or 'quad'

# Experiment
integration_time = DAY_TO_SEC       # Integration time for transient search [s]
integration_time_DM = YEAR_TO_SEC   # Integration time for DM search [s]
sensitivity = 1e-17                 # Fractional frequency sensitivity

# Constraint plot
xlims = (1e-21, 6e-6)        # Mass axis range [eV]
ylims = (5e-12, 2e-8)        # Coupling axis range

In [ ]:
# Define source parameters
source = Source(
    Etot=Etot,
    mass=mass,
    tstar=tstar,
    R=R,
    ULB_type=ULB_type,
    coupling_type=coupling_type,
    coupling_order=coupling_order
)

print(f'Source configuration:')
print(f'  mass = {source.mass} eV')
print(f'  R = {source.R} pc')
print(f'  tstar = {source.tstar} s')
print(f'  Etot = {source.Etot / 1.989e66:.4f} solar masses')
print(f'  coupling_type = {source.coupling_type}')
print(f'  coupling_order = {source.coupling_order}')

#### Configure Experiment and Generate Constraint Plot

In [ ]:
# Define experiment parameters
experiment = Experiment(
    integration_time=integration_time,
    integration_time_DM=integration_time_DM,
    sensitivity=sensitivity,
    time_delays={'day': DAY_TO_SEC, 'year': YEAR_TO_SEC}
)

# Create signal model
signal_model = SignalModel(source=source, experiment=experiment)

print('Signal model created successfully')

In [ ]:
# Generate constraint plot
plot = Plot(
    xlims=xlims,
    ylims=ylims,
    exclude_mass=False,
    include_legend=True
)

output = OutputHandler(plot)
output.plot_parameter_space(
    source, 
    signal_model, 
    plot, 
    save_path='plots/constraints_manual.png',
)

print('Constraint plot saved to plots/constraints_manual.png')

## Example 2: 2x2 Parameter Grid

Create a grid of constraint plots varying mass and burst duration:

In [ ]:
# --- Example 2 Parameters ---
masses = [1e-20, 1e-15]      # Two different masses [eV]
times = [1, 100]             # Two different burst durations [seconds]
Etot_grid = 0.01             # Total energy [solar masses]
R_grid = 1e4                 # Distance [parsecs]
ULB_type_grid = 'ALP'        # Type: 'scalar' or 'ALP'
coupling_type_grid = 'electron'  # Coupling type

# Constraint plot
xlims_grid = (1e-21, 6e-6)   # Mass axis range [eV]
ylims_grid = (1e-13, 2e-8)   # Coupling axis range
legend_bbox = (0.88, 0.88)   # Legend anchor position

In [ ]:
# Create 2x2 grid of sources
sources_grid = [
    [Source(
        Etot=Etot_grid, 
        mass=m, 
        tstar=t, 
        R=R_grid, 
        ULB_type=ULB_type_grid, 
        coupling_type=coupling_type_grid
    ) for m in masses] 
    for t in times
]

# Create corresponding signal models
signal_models_grid = [
    [SignalModel(source=s, experiment=experiment) for s in row] 
    for row in sources_grid
]

print(f'Created 2x2 grid with:')
print(f'  Masses: {masses} eV')
print(f'  Burst durations: {times} s')

In [ ]:
# Configure plot with custom legend position
legend_config = {
    'frameon': False,
    'bbox_to_anchor': legend_bbox
}

plot_2x2 = Plot(
    xlims=xlims_grid, 
    ylims=ylims_grid, 
    exclude_mass=False, 
    include_legend=True, 
    legend_config=legend_config
)

output_2x2 = OutputHandler(plot_2x2)
output_2x2.plot_parameter_space(
    sources_grid, 
    signal_models_grid, 
    plot_2x2, 
    save_path='plots/constraints_2x2.png'
)

print('2x2 constraint plot saved to plots/constraints_2x2.png')

---

## Optional: Load from Previously Saved Parameters

If you ran `examples_complete_workflow.ipynb` and exported parameters, you can load them here:

In [ ]:
from waveform.propagation import load_source_from_file

# Load source from parameter file (if it exists)
try:
    source_from_file = load_source_from_file('bosenova.param', ULB_type='scalar')
    
    print('Loaded source from bosenova.param:')
    print(f'  mass = {source_from_file.mass} eV')
    print(f'  R = {source_from_file.R} pc')
    print(f'  tstar = {source_from_file.tstar} s')
    print(f'  Etot = {source_from_file.Etot / 1.989e66:.4f} solar masses')
    print(f'  coupling_type = {source_from_file.coupling_type}')
    print(f'  coupling_order = {source_from_file.coupling_order}')
    
    # Generate constraint plot with loaded parameters
    signal_model_from_file = SignalModel(source=source_from_file, experiment=experiment)
    output.plot_parameter_space(
        source_from_file, 
        signal_model_from_file, 
        plot, 
        save_path='plots/constraints_from_file.png'
    )
    
    print('\nConstraint plot saved to plots/constraints_from_file.png')
    
except FileNotFoundError:
    print('No parameter file found. Run examples_complete_workflow.ipynb first to generate bosenova.param')

---

## Summary

This notebook demonstrated manual parameter exploration with:
1. **Example 1: Single source (1x1 plot)** - Basic constraint plot generation
2. **Example 2: Parameter grid (2x2 plot)** - Varying mass and burst duration
3. **Optional file loading** - Resume from propagation results

For the **complete workflow** including spectrogram generation, see `examples_complete_workflow.ipynb`.